# 01 Data Generation or Ingestion

This notebook generates synthetic Type 1 diabetes patient and glucose data for the analytics project.

Outputs:
- data/raw/patients.csv
- data/raw/glucose_readings.csv

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta

In [5]:
rng = np.random.default_rng(seed=42)

project_root = Path.cwd().parent
raw_data_path = project_root / "data" / "raw"
raw_data_path.mkdir(parents=True, exist_ok=True)

print("Raw data path:", raw_data_path)

Raw data path: c:\Users\Joshua\Documents\type1-diabetes-analytics\data\raw


In [6]:
n_patients = 10
start_date = pd.Timestamp("2026-01-01")
num_days = 7
glucose_freq_minutes = 30

In [8]:
sexes = ["Female", "Male"]
age_groups = ["18-24", "25-34", "35-44", "45-54", "55-64"]
insulin_methods = ["Pump", "Multiple Daily Injections"]
cgm_devices = ["Dexcom", "Libre", "Medtronic"]

patients = pd.DataFrame({
    "patient_id": range(1, n_patients + 1),
    "sex": rng.choice(sexes, n_patients),
    "age_group": rng.choice(age_groups, n_patients),
    "diabetes_duration_years": rng.integers(1, 31, n_patients),
    "insulin_delivery_method": rng.choice(insulin_methods, n_patients),
    "cgm_device_type": rng.choice(cgm_devices, n_patients)
})

patients.head()

,patient_id,sex,age_group,diabetes_duration_years,insulin_delivery_method,cgm_device_type
0,1,Male,35-44,5,Multiple Daily Injections,Medtronic
1,2,Female,25-34,23,Pump,Medtronic
2,3,Female,18-24,22,Pump,Libre
3,4,Male,35-44,11,Pump,Medtronic
4,5,Male,55-64,3,Pump,Libre


In [9]:
timestamps = pd.date_range(
    start=start_date,
    periods=(24 * 60 // glucose_freq_minutes) * num_days,
    freq=f"{glucose_freq_minutes}min"
)

len(timestamps), timestamps[:5]

(336,
 DatetimeIndex(['2026-01-01 00:00:00', '2026-01-01 00:30:00',
                '2026-01-01 01:00:00', '2026-01-01 01:30:00',
                '2026-01-01 02:00:00'],
               dtype='datetime64[us]', freq='30min'))

In [10]:
glucose_rows = []
measurement_id = 1

for patient_id in patients["patient_id"]:
    baseline = rng.normal(145, 20)
    variability = rng.uniform(15, 40)

    for ts in timestamps:
        hour = ts.hour

        # Mild daily pattern
        if 6 <= hour <= 9:
            meal_effect = rng.normal(15, 10)
        elif 12 <= hour <= 14:
            meal_effect = rng.normal(20, 12)
        elif 18 <= hour <= 21:
            meal_effect = rng.normal(25, 15)
        elif 0 <= hour <= 5:
            meal_effect = rng.normal(-10, 8)
        else:
            meal_effect = rng.normal(0, 8)

        glucose_value = baseline + meal_effect + rng.normal(0, variability)
        glucose_value = max(40, min(400, round(glucose_value, 2)))

        glucose_rows.append({
            "measurement_id": measurement_id,
            "patient_id": patient_id,
            "measurement_timestamp": ts,
            "glucose_mg_dl": glucose_value
        })

        measurement_id += 1

glucose = pd.DataFrame(glucose_rows)
glucose.head()

,measurement_id,patient_id,measurement_timestamp,glucose_mg_dl
0,1,1,2026-01-01 00:00:00,135.81
1,2,1,2026-01-01 00:30:00,176.35
2,3,1,2026-01-01 01:00:00,149.78
3,4,1,2026-01-01 01:30:00,161.57
4,5,1,2026-01-01 02:00:00,175.23


In [11]:
print("Patients shape:", patients.shape)
print("Glucose shape:", glucose.shape)

print("\nPatients preview:")
display(patients.head())

print("\nGlucose preview:")
display(glucose.head())

print("\nGlucose summary:")
display(glucose["glucose_mg_dl"].describe())

Patients shape: (10, 6)
Glucose shape: (3360, 4)

Patients preview:


,patient_id,sex,age_group,diabetes_duration_years,insulin_delivery_method,cgm_device_type
0,1,Male,35-44,5,Multiple Daily Injections,Medtronic
1,2,Female,25-34,23,Pump,Medtronic
2,3,Female,18-24,22,Pump,Libre
3,4,Male,35-44,11,Pump,Medtronic
4,5,Male,55-64,3,Pump,Libre



Glucose preview:


,measurement_id,patient_id,measurement_timestamp,glucose_mg_dl
0,1,1,2026-01-01 00:00:00,135.81
1,2,1,2026-01-01 00:30:00,176.35
2,3,1,2026-01-01 01:00:00,149.78
3,4,1,2026-01-01 01:30:00,161.57
4,5,1,2026-01-01 02:00:00,175.23



Glucose summary:


count    3360.000000
mean      141.264295
std        38.036506
min        40.000000
25%       116.055000
50%       141.830000
75%       168.215000
max       277.360000
Name: glucose_mg_dl, dtype: float64

In [12]:
print("Nulls in patients:")
print(patients.isnull().sum())

print("\nNulls in glucose:")
print(glucose.isnull().sum())

print("\nDuplicate patient IDs:", patients["patient_id"].duplicated().sum())
print("Duplicate measurement IDs:", glucose["measurement_id"].duplicated().sum())

Nulls in patients:
patient_id                 0
sex                        0
age_group                  0
diabetes_duration_years    0
insulin_delivery_method    0
cgm_device_type            0
dtype: int64

Nulls in glucose:
measurement_id           0
patient_id               0
measurement_timestamp    0
glucose_mg_dl            0
dtype: int64

Duplicate patient IDs: 0
Duplicate measurement IDs: 0


In [13]:
patients.to_csv(raw_data_path / "patients.csv", index=False)
glucose.to_csv(raw_data_path / "glucose_readings.csv", index=False)

print("Saved:")
print(raw_data_path / "patients.csv")
print(raw_data_path / "glucose_readings.csv")

Saved:
c:\Users\Joshua\Documents\type1-diabetes-analytics\data\raw\patients.csv
c:\Users\Joshua\Documents\type1-diabetes-analytics\data\raw\glucose_readings.csv
